## Ai load

In [1]:
import os 
from dotenv import load_dotenv, find_dotenv
_=load_dotenv(find_dotenv())
groq_api_key = os.getenv("GROQ_API_KEY")


## LLM connnect

In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile")


## Information Extract

In [3]:
from typing import Optional

from pydantic import BaseModel, Field

class Person(BaseModel):
    """information about a person"""

    name: Optional[str] = Field(
        default=None, description="the person's name"
    )
    last_name: Optional[str] = Field(
        default=None, description="the person's last name"
    )
    country: Optional[str] = Field(
        default=None, description="the country of the person if known"
    )
    

## Define the Extractor

In [4]:
from typing import Optional

from langchain_core.prompts import ChatPromptTemplate
from  pydantic import BaseModel, Field


prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "you are an extraction algorithm."
     "only exctract relavant information from the text."
     "if you do not know the value of an attribute asked the extract."
     "return null from the attribute's value. ",
      
    ),
    ("human", "{text}"),
  ]
)


In [5]:
chain = prompt | llm.with_structured_output(schema=Person)


In [6]:
comment = "I absolutely love this product! it's been game-changer for my daily routine. the quality is top-notch and the customer service is outstanding. i higlhy recommend it to all my friands and family. sarah-smith, USA"


In [7]:
chain.invoke({"text": comment})


Person(name='sarah-smith', last_name='smith', country='USA')

## Multiple person

In [8]:
from typing import Optional, List

from pydantic import BaseModel, Field

class Person(BaseModel):
    """information about a person"""

    name: Optional[str] = Field(
        default=None, description="the person's name"
    )
    last_name: Optional[str] = Field(
        default=None, description="the person's last name"
    )
    country: Optional[str] = Field(
        default=None, description="the country of the person if known"
    )

class Data(BaseModel):
    """Extracted data from the text"""

    people: List[Person]
    

In [9]:
chain = prompt | llm.with_structured_output(schema=Data)


In [10]:
comment = "i'm so impressed with the product! it's been a game-changer for my daily routine. the quality is top-notch and the customer service is outstanding. i highly recommend it to all my friends and family. john-doe, USA; jane-smith, UK"


In [11]:
chain.invoke({"text": comment})


Data(people=[Person(name='John-Doe', last_name='Doe', country='USA'), Person(name='Jane-Smith', last_name='Smith', country='UK')])

In [12]:
text_input = """
 
Alice Johnson, a software engineer from the USA, has been working on AI projects for over 5 years. and Bob Smith, a data scientist from the UK, has a background in machine learning and has contributed to several open-source projects. both of them are passionate about technology and innovation.
"""
response = chain.invoke({"text": text_input})


response



Data(people=[Person(name='Alice Johnson', last_name='Johnson', country='USA'), Person(name='Bob Smith', last_name='Smith', country='UK')])